In [1]:
import argparse
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from dataclasses import dataclass
from torchvision.utils import make_grid, save_image

In [2]:
def per_sample_entropy(z):
    """
    z: [B, L, V]
    returns: [B] entropy averaged over groups
    """
    z_safe = z.clamp_min(1e-8)
    H = -(z_safe * z_safe.log()).sum(dim=-1)  # [B, L]
    return H.mean(dim=1)                      # [B]


def batch_marginal_entropy(z):
    """
    z: [B, L, V]
    returns: [L] entropy of batch-averaged distribution per group
    """
    p_bar = z.mean(dim=0)  # [L, V]
    p_safe = p_bar.clamp_min(1e-8)
    H = -(p_safe * p_safe.log()).sum(dim=-1)  # [L]
    return H


class SEMHead(nn.Module):
    def __init__(self, in_dim, L, V, tau=1.0):
        super().__init__()
        self.L = L
        self.V = V
        self.tau = tau
        self.lin = nn.Linear(in_dim, L * V)

    def forward(self, h, tau=None, noise_std=0.0):
        tau = self.tau if tau is None else tau
        logits = self.lin(h).view(h.size(0), self.L, self.V)  # [B, L, V]
        z = F.softmax(logits / tau, dim=-1)                   # [B, L, V]

        if noise_std > 0.0:
            z = z + noise_std * torch.randn_like(z)

        return z, logits

    def hard_codes(self, logits):
        """
        logits: [B, L, V]
        returns:
          z_hard: [B, L, V] one-hot
          idx: [B, L] indices
        """
        idx = logits.argmax(dim=-1)  # [B, L]
        z_hard = F.one_hot(idx, num_classes=self.V).float()
        return z_hard, idx


class SEMAutoencoder(nn.Module):
    def __init__(self, x_dim=28 * 28, hidden_dim=512, bottleneck_dim=256,
                 L=8, V=16, tau=1.0, big_decoder=False):
        super().__init__()
        self.x_dim = x_dim
        self.encoder = nn.Sequential(
            nn.Linear(x_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, bottleneck_dim),
            nn.ReLU(),
        )
        self.sem = SEMHead(bottleneck_dim, L, V, tau=tau)
        if big_decoder:
            self.decoder = nn.Sequential(
                nn.Linear(L * V, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim*4),
                nn.ReLU(),
                nn.Linear(hidden_dim*4, hidden_dim*8),
                nn.ReLU(),
                nn.Linear(hidden_dim*8, hidden_dim*8),
                nn.ReLU(),
                nn.Linear(hidden_dim*8, hidden_dim*4),
                nn.ReLU(),
                nn.Linear(hidden_dim*4, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, x_dim),
                nn.Sigmoid(),  # MNIST pixels in [0,1]
            )
        else:
            self.decoder = nn.Sequential(
                nn.Linear(L * V, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, x_dim),
                nn.Sigmoid(),  # MNIST pixels in [0,1]
            )

    def forward(self, x, tau=None, noise_std=0.0):
        # x: [B, x_dim]
        h = self.encoder(x)
        z, logits = self.sem(h, tau=tau, noise_std=noise_std)
        y = self.decoder(z.view(z.size(0), -1))
        return y, z, logits
    
def visualize(model, test_loader, device, args):
    model.eval()
    x, _ = next(iter(test_loader))
    x = x.to(device)  # [B, 784]

    # Use final tau, no noise for visualization
    tau = args.tau_final
    noise_std = 0.0

    with torch.no_grad():
        # soft recon
        y_soft, z, logits = model(x, tau=tau, noise_std=noise_std)

        # hard recon via argmax codes
        z_hard, _ = model.sem.hard_codes(logits)        # [B, L, V]
        y_hard = model.decoder(z_hard.view(z_hard.size(0), -1))

    # reshape to images
    B = min(args.n_vis, x.size(0))
    x_img = x[:B].view(B, 1, 28, 28)
    y_soft_img = y_soft[:B].view(B, 1, 28, 28)
    y_hard_img = y_hard[:B].view(B, 1, 28, 28)

    # stack: row1 = original, row2 = soft recon, row3 = hard recon
    grid = torch.cat([x_img, y_soft_img, y_hard_img], dim=0)  # [3B, 1, 28, 28]
    grid = make_grid(grid, nrow=B, padding=2)

    os.makedirs(args.out_dir, exist_ok=True)
    out_path = os.path.join(args.out_dir, f"vis_sem_ae_{args.mode}.png")
    save_image(grid, out_path)
    print(f"Saved visualization grid to {out_path}")

def evaluate_test_loss(model, test_loader, device, args):
    model.eval()
    total_soft = 0.0
    total_hard = 0.0
    n = 0

    with torch.no_grad():
        for x, _ in test_loader:
            x = x.to(device)
            B = x.size(0)

            # use final temperature and no noise
            tau = args.tau_final

            # soft reconstruction
            y_soft, z, logits = model(x, tau=tau, noise_std=0.0)
            if args.bce_loss:
                loss_soft = F.binary_cross_entropy(y_soft, x, reduction="sum").item()
            else:
                loss_soft = F.mse_loss(y_soft, x, reduction="sum").item()

            # hard reconstruction
            z_hard, _ = model.sem.hard_codes(logits)
            y_hard = model.decoder(z_hard.view(B, -1))
            if args.bce_loss:
                loss_hard = F.binary_cross_entropy(y_hard, x, reduction="sum").item()
            else:
                loss_hard = F.mse_loss(y_hard, x, reduction="sum").item()

            total_soft += loss_soft
            total_hard += loss_hard
            n += B

    return total_soft / n, total_hard / n

In [31]:
1/16

0.0625

In [32]:
@dataclass
class CFG:
    mode: str = 'noise'
    L: int = 8
    V: int = 16
    tau_init: float = 1.0
    tau_final: float = 0.3
    hidden_dim: int = 512
    bottleneck_dim: int = 256
    lambda_H_max: float = 0.05
    noise_std_max: float = 0.1
    epochs: int = 5
    batch_size: int = 256
    lr: float = 1e-3
    bce_loss: bool = True
    data_dir: str = "./toy/data"
    out_dir: str = "./toy/checkpoints"
    save_model: bool = False
    log_every: int = 200
    cpu: bool = False
    visualize: bool = True
    n_vis: int = 3
    big_decoder: bool = False

In [47]:
def train(args : CFG):
    codes = []

    device = "cuda" if torch.cuda.is_available() and not args.cpu else "cpu"
    print(f"Using device: {device}")

    # MNIST data
    transform = transforms.Compose([
        transforms.ToTensor(),     # [0,1]
        transforms.Lambda(lambda t: t.view(-1))  # flatten 28x28 -> 784
    ])
    train_ds = datasets.MNIST(
        root=args.data_dir, train=True, download=True, transform=transform
    )
    test_ds = datasets.MNIST(
        root=args.data_dir, train=False, download=True, transform=transform
    )

    train_loader = DataLoader(
        train_ds, batch_size=args.batch_size, shuffle=True, num_workers=4, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=args.batch_size, shuffle=False, num_workers=4, pin_memory=True
    )

    model = SEMAutoencoder(
        x_dim=28 * 28,
        hidden_dim=args.hidden_dim,
        bottleneck_dim=args.bottleneck_dim,
        L=args.L, V=args.V,
        tau=args.tau_init,
        big_decoder=args.big_decoder
    ).to(device)

    opt = torch.optim.Adam(model.parameters(), lr=args.lr)

    # total steps for schedule
    steps_per_epoch = math.ceil(len(train_loader) / 1.0)
    total_steps = args.epochs * steps_per_epoch

    global_step = 0

    for epoch in range(args.epochs):
        model.train()
        for x, _ in train_loader:
            x = x.to(device)

            # fraction of training completed
            frac = min(global_step / max(1, total_steps), 1.0)

            # schedules
            if args.mode in ["entropy", "both"]:
                lambda_H = args.lambda_H_max * frac
            else:
                lambda_H = 0.0

            if args.mode in ["noise", "both"]:
                noise_std = args.noise_std_max #* frac
            else:
                noise_std = 0.0

            # temperature annealing (optional)
            tau = args.tau_init - frac * (args.tau_init - args.tau_final)
            tau = max(tau, args.tau_final)

            # forward
            y, z, logits = model(x, tau=tau, noise_std=noise_std)

            # reconstruction loss (MSE or BCE)
            if args.bce_loss:
                recon_loss = F.binary_cross_entropy(y, x, reduction="mean")
            else:
                recon_loss = F.mse_loss(y, x, reduction="mean")

            H_sample = per_sample_entropy(logits.softmax(-1)).mean() / math.log(args.V)

            loss = recon_loss + lambda_H * H_sample

            opt.zero_grad()
            loss.backward()
            opt.step()

            if global_step % args.log_every == 0:
                with torch.no_grad():
                    codes.append(logits)

                    H_marg = batch_marginal_entropy(logits.softmax(-1)).mean().item() / math.log(args.V)

                    # hard reconstruction (argmax codes)
                    z_hard, _ = model.sem.hard_codes(logits)
                    
                    y_hard = model.decoder(z_hard.view(z_hard.size(0), -1))
                    if args.bce_loss:
                        hard_recon = F.binary_cross_entropy(y_hard, x, reduction="mean").item()
                    else:
                        hard_recon = F.mse_loss(y_hard, x, reduction="mean").item()

                # evaluate on test set
                test_soft, test_hard = evaluate_test_loss(model, test_loader, device, args)
                
                print(
                    f"[epoch {epoch:02d} step {global_step:06d}] "
                    f"mode={args.mode} "
                    f"loss={loss.item():.4f} "
                    f"val_loss={test_soft:.4f} "
                    f"recon={recon_loss.item():.4f} "
                    f"recon_hard={hard_recon:.4f} "
                    f"H_sample={H_sample.item():.3f} "
                    f"H_marg={H_marg:.3f} "
                    f"tau={tau:.3f} "
                    f"lambda_H={lambda_H:.4f} "
                    f"noise={noise_std:.3f}"
                )

            global_step += 1

    


    # save model if asked
    os.makedirs(args.out_dir, exist_ok=True)
    if args.save_model:
        torch.save(model.state_dict(), os.path.join(args.out_dir, f"sem_ae_{args.mode}.pt"))
        print("Model saved.")

    # visualization step
    if args.visualize:
        visualize(model, test_loader, device, args)
    return codes

In [48]:
xd = train(CFG(tau_final=1.0, L=16, V=16, epochs=10, noise_std_max=2/16, mode='noise', big_decoder=True))

Using device: cuda
[epoch 00 step 000000] mode=noise loss=0.6933 val_loss=535.1535 recon=0.6933 recon_hard=0.6769 H_sample=1.000 H_marg=1.000 tau=1.000 lambda_H=0.0000 noise=0.125
[epoch 00 step 000200] mode=noise loss=0.2327 val_loss=186.8670 recon=0.2327 recon_hard=0.2616 H_sample=0.606 H_marg=0.751 tau=1.000 lambda_H=0.0000 noise=0.125
[epoch 01 step 000400] mode=noise loss=0.2057 val_loss=160.5707 recon=0.2057 recon_hard=0.2134 H_sample=0.391 H_marg=0.706 tau=1.000 lambda_H=0.0000 noise=0.125
[epoch 02 step 000600] mode=noise loss=0.1716 val_loss=130.4732 recon=0.1716 recon_hard=0.1773 H_sample=0.296 H_marg=0.630 tau=1.000 lambda_H=0.0000 noise=0.125
[epoch 03 step 000800] mode=noise loss=0.1542 val_loss=118.3571 recon=0.1542 recon_hard=0.1646 H_sample=0.303 H_marg=0.656 tau=1.000 lambda_H=0.0000 noise=0.125
[epoch 04 step 001000] mode=noise loss=0.1473 val_loss=112.5223 recon=0.1473 recon_hard=0.1608 H_sample=0.306 H_marg=0.677 tau=1.000 lambda_H=0.0000 noise=0.125
[epoch 05 step 

In [41]:
2/16

0.125